In [ ]:
pip install requests

In [ ]:
"""
Sube una estructura de carpetas local completa a un repositorio de GitHub,
manteniendo la jerarquía de carpetas y subcarpetas.

Requisitos:
    pip install requests

Uso:
    python subir_a_github.py

Antes de ejecutar, completá las variables de configuración de más abajo.
"""

import os
import base64
import requests
from google.colab import drive
import os

drive.mount('/content/drive')

try:
    from google.colab import userdata
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
except Exception:
    import getpass
    GITHUB_TOKEN = getpass.getpass("Pegá tu GitHub Personal Access Token: ")



print("✅ README.md ampliado generado correctamente.")



# ============ CONFIGURACIÓN ============
OWNER = "robertodavidalcoba-design"       # Usuario u organización dueño del repo
REPO = "Curso-DataAnalitic-CoderHouse"          # Nombre del repositorio
BRANCH = "main"                           # Rama destino
LOCAL_FOLDER = "/content/drive/MyDrive/Trabajo Final Data2"    # Carpeta local que querés subir
REMOTE_BASE_PATH = ""                     # Carpeta destino dentro del repo ("" = raíz)

%cd {LOCAL_FOLDER}
contenido="""# Proyecto Data Analysis en Google Colab

## Descripción
Este repositorio contiene una estructura reproducible pensada para ejecutarse directamente en Google Colab o entornos de Jupyter.

## Estructura del Repositorio
```text
project-root/
│
├── README.md           # Documentación del proyecto
├── requirements.txt    # Librerías necesarias
├── notebooks/          # Notebooks ordenados por etapa
├── scripts/            # Módulos Python reutilizables (utils.py)
├── data/               # Datasets de entrada
└── outputs/            # Gráficos y reportes generados

"""

with open("README.md", "w") as f:
    f.write(contenido)
f.close()


# Carpetas/archivos a ignorar
IGNORAR = {".git", "__pycache__", ".DS_Store", "node_modules", ".venv"}
# ========================================

API_URL = "https://api.github.com"

HEADERS = {
    "Authorization": f"token {GITHUB_TOKEN}",
    "Accept": "application/vnd.github+json",
}


def obtener_sha_existente(ruta_remota):
    """Si el archivo ya existe en el repo, devuelve su sha (necesario para actualizarlo)."""
    url = f"{API_URL}/repos/{OWNER}/{REPO}/contents/{ruta_remota}"
    resp = requests.get(url, headers=HEADERS, params={"ref": BRANCH})
    if resp.status_code == 200:
        return resp.json().get("sha")
    return None


def subir_archivo(ruta_local, ruta_remota):
    """Sube (o actualiza) un archivo individual al repositorio."""
    with open(ruta_local, "rb") as f:
        contenido = f.read()

    contenido_b64 = base64.b64encode(contenido).decode("utf-8")
    sha = obtener_sha_existente(ruta_remota)

    data = {
        "message": f"Subir {ruta_remota}",
        "content": contenido_b64,
        "branch": BRANCH,
    }
    if sha:
        data["sha"] = sha  # requerido si el archivo ya existe

    url = f"{API_URL}/repos/{OWNER}/{REPO}/contents/{ruta_remota}"
    resp = requests.put(url, headers=HEADERS, json=data)

    if resp.status_code in (200, 201):
        print(f"✅ Subido: {ruta_remota}")
    else:
        print(f"❌ Error en {ruta_remota}: {resp.status_code} - {resp.json().get('message')}")


def recorrer_y_subir(carpeta_local, base_remota):
    """Recorre recursivamente la carpeta local y sube cada archivo respetando la jerarquía."""
    for raiz, carpetas, archivos in os.walk(carpeta_local):
        carpetas[:] = [c for c in carpetas if c not in IGNORAR]

        for archivo in archivos:
            if archivo in IGNORAR:
                continue

            ruta_local = os.path.join(raiz, archivo)
            ruta_relativa = os.path.relpath(ruta_local, carpeta_local)
            ruta_relativa = ruta_relativa.replace(os.sep, "/")  # rutas estilo GitHub

            ruta_remota = f"{base_remota}/{ruta_relativa}" if base_remota else ruta_relativa

            subir_archivo(ruta_local, ruta_remota)


if __name__ == "__main__":
    if not GITHUB_TOKEN:
        print("⚠️  No se encontró el token. Configurá el secreto GITHUB_TOKEN en Colab (🔑) o pegalo cuando se te pida.")
    elif not os.path.exists(LOCAL_FOLDER):
        print(f"❌ La carpeta '{LOCAL_FOLDER}' no existe. Verificá la ruta (ruta absoluta recomendada).")
    else:
        # Contar archivos antes de subir, para detectar carpetas vacías o mal apuntadas
        total_archivos = sum(
            1 for _, _, archivos in os.walk(LOCAL_FOLDER) for a in archivos if a not in IGNORAR
        )
        if total_archivos == 0:
            print(f"⚠️  No se encontró ningún archivo dentro de '{LOCAL_FOLDER}'. Nada para subir.")
        else:
            print(f"📁 Se encontraron {total_archivos} archivo(s) en '{LOCAL_FOLDER}'. Subiendo...")
            recorrer_y_subir(LOCAL_FOLDER, REMOTE_BASE_PATH)
            print("🎉 Proceso terminado.")

Mounted at /content/drive
